# Supervisor and Hierarchical Agent Patterns
Companion notebook for the **Supervisor and Hierarchical Agent Patterns** section of Chapter 8.

One supervisor stays in charge: it breaks a request into subtasks, assigns each to a focused worker, tracks progress, and merges the results. The pattern is **split, assign, collect, merge**.

The book leaves `run_worker` and `merge_results` as given -- here they're small, deterministic stand-ins for the intake/risk/decision specialists from the chapter's loan-review example, so the flow below actually runs end to end.

In [1]:
def run_worker(worker: str, input_data: dict) -> dict:
    """Stand-in for a real specialist agent call -- deterministic, so the demo is repeatable."""
    if worker == "intake":
        return {"validated": True, "applicant": input_data["name"]}
    if worker == "risk":
        score = 0.2 if input_data["credit_score"] >= 700 else 0.7
        return {"risk_score": score, "flag": "low" if score < 0.5 else "high"}
    raise ValueError(f"no worker registered for role: {worker}")


def merge_results(results: list[dict]) -> dict:
    by_task = {r["task_id"]: r["result"] for r in results}
    return {
        "applicant": by_task["t1"]["applicant"],
        "risk_flag": by_task["t2"]["flag"],
        "decision": "approve" if by_task["t2"]["flag"] == "low" else "manual_review",
    }

This is the book's own supervisor loop, unchanged -- explicit task IDs and worker assignments, not hidden chat history:

In [2]:
app_data = {"name": "Asha Mehta", "credit_score": 742}
policy_data = {"name": "Asha Mehta", "credit_score": 742}

tasks = [
    {"id": "t1", "worker": "intake", "input": app_data},
    {"id": "t2", "worker": "risk", "input": policy_data},
]

results = []
for task in tasks:
    result = run_worker(task["worker"], task["input"])
    results.append({"task_id": task["id"], "result": result})

final_answer = merge_results(results)
print(final_answer)

{'applicant': 'Asha Mehta', 'risk_flag': 'low', 'decision': 'approve'}


Keeping `results` as an explicit list (task_id + worker output), rather than folding everything into one prompt, is what makes this replayable: you can see exactly which worker produced which fact, add a `status`/`retries` field per task, and re-run just the failed one instead of the whole flow.

In [3]:
# Extending the same loop with status tracking -- the "explicit state" the book calls for
task_state = []
for task in tasks:
    entry = {"task_id": task["id"], "worker": task["worker"], "status": "running", "retries": 0}
    task_state.append(entry)
    entry["result"] = run_worker(task["worker"], task["input"])
    entry["status"] = "done"

for entry in task_state:
    print(f"{entry['task_id']} ({entry['worker']}): status={entry['status']} result={entry['result']}")

t1 (intake): status=done result={'validated': True, 'applicant': 'Asha Mehta'}
t2 (risk): status=done result={'risk_score': 0.2, 'flag': 'low'}
